# Protótipo Funcional: Detecção Comportamental de Anomalias em APIs com IA e Resposta SOAR
**Instituição:** SENAI / UniSENAI — CST em Defesa Cibernética  
**Autor:** Vinicius Ocker Fagundes  
**Objetivo:** Simular a ingestão de telemetria normalizada (ECS), executar detecção não supervisionada via `IsolationForest` considerando a sazonalidade fora do horário comercial e acionar a matriz de decisão proporcional do `Shuffle SOAR`.

In [1]:
import json
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

rng = np.random.default_rng(42)
n_normal, n_attack = 2000, 80

# 1. Telemetria legítima (distribuição horária comercial predominante)
weights = np.array([
    0.01, 0.005, 0.005, 0.005, 0.01, 0.02, 0.04, 0.06,
    0.08, 0.09, 0.09, 0.08, 0.07, 0.08, 0.08, 0.07,
    0.06, 0.05, 0.04, 0.025, 0.015, 0.01, 0.005, 0.005
])
weights = weights / weights.sum()

hour_normal = rng.choice(np.arange(0, 24), size=n_normal, p=weights)
off_hours_normal = ((hour_normal < 8) | (hour_normal >= 20)).astype(int)
req_rate_normal = rng.normal(loc=14.0, scale=4.5, size=n_normal).clip(1, 35)
payload_entropy_normal = rng.normal(loc=3.6, scale=0.35, size=n_normal).clip(2.0, 4.6)
error_4xx_ratio_normal = rng.beta(a=1.2, b=18.0, size=n_normal)
unique_endpoints_normal = rng.poisson(lam=2.2, size=n_normal).clip(1, 8)

# 2. Ataques automatizados fora do expediente (Credential Stuffing, API Scraping, BOLA)
hour_attack = rng.choice([0, 1, 2, 3, 4, 5, 22, 23], size=n_attack)
off_hours_attack = np.ones(n_attack, dtype=int)
req_rate_attack = rng.normal(loc=92.0, scale=18.0, size=n_attack).clip(45, 160)
payload_entropy_attack = rng.normal(loc=5.4, scale=0.45, size=n_attack).clip(4.3, 6.8)
error_4xx_ratio_attack = rng.beta(a=8.0, b=3.0, size=n_attack)
unique_endpoints_attack = rng.poisson(lam=14.0, size=n_attack).clip(6, 30)

df = pd.DataFrame({
    'hour': np.concatenate([hour_normal, hour_attack]),
    'off_hours_flag': np.concatenate([off_hours_normal, off_hours_attack]),
    'req_rate_1m': np.round(np.concatenate([req_rate_normal, req_rate_attack]), 2),
    'payload_entropy': np.round(np.concatenate([payload_entropy_normal, payload_entropy_attack]), 3),
    'error_4xx_ratio': np.round(np.concatenate([error_4xx_ratio_normal, error_4xx_ratio_attack]), 3),
    'unique_endpoints_5m': np.concatenate([unique_endpoints_normal, unique_endpoints_attack]),
    'is_attack': np.concatenate([np.zeros(n_normal, dtype=int), np.ones(n_attack, dtype=int)])
})
df.head()

   hour  off_hours_flag  req_rate_1m  payload_entropy  error_4xx_ratio  unique_endpoints_5m  is_attack  anomaly_score                              soar_action  predicted_alert
0    15               0         9.39            2.899            0.045                    1          0         0.2083  BAIXO -> Log ECS & Atualizacao Baseline                0
1    11               0        18.09            3.334            0.174                    1          0         0.2879  BAIXO -> Log ECS & Atualizacao Baseline                0
2    17               0         8.59            3.944            0.033                    3          0         0.1179  BAIXO -> Log ECS & Atualizacao Baseline                0
3    14               0        12.71            3.380            0.031                    1          0         0.0265  BAIXO -> Log ECS & Atualizacao Baseline                0
4     6               1        11.36            3.310            0.056                    4          0         0.3670  B

In [2]:
features = ['off_hours_flag', 'req_rate_1m', 'payload_entropy', 'error_4xx_ratio', 'unique_endpoints_5m']

# Treinamento não supervisionado com baseline de tráfego legítimo
model = IsolationForest(n_estimators=200, max_samples=256, contamination=0.038, random_state=42)
model.fit(df[df['is_attack'] == 0][features])

# Cálculo do Anomaly Score normalizado (0.00 a 1.00)
raw_scores = -model.decision_function(df[features])
df['anomaly_score'] = np.round((raw_scores - raw_scores.min()) / (raw_scores.max() - raw_scores.min()), 4)

# Matriz de decisão proporcional do Shuffle SOAR
def soar_decision(row):
    if row['anomaly_score'] >= 0.85:
        return 'ALTO/CRITICO -> Bloqueio WAF (TTL 24h) + Revogar Token OAuth2 + Caso TheHive'
    elif row['anomaly_score'] >= 0.65:
        return 'MEDIO -> Rate Limiting Adaptativo + Step-Up Auth (MFA/CAPTCHA)'
    return 'BAIXO -> Log ECS & Atualizacao Baseline'

df['soar_action'] = df.apply(soar_decision, axis=1)
df['predicted_alert'] = (df['anomaly_score'] >= 0.65).astype(int)

precision = precision_score(df['is_attack'], df['predicted_alert'])
recall = recall_score(df['is_attack'], df['predicted_alert'])
f1 = f1_score(df['is_attack'], df['predicted_alert'])
cm = confusion_matrix(df['is_attack'], df['predicted_alert'])
fp_rate = cm[0, 1] / (cm[0, 0] + cm[0, 1])

print('=== RESULTADOS DA EXECUÇÃO DO PROTÓTIPO (ISOLATION FOREST + SOAR) ===')
print(f'Total de eventos processados: {len(df)} (Normais: {n_normal} | Ataques: {n_attack})')
print(f'Precision: {precision:.4f} | Recall: {recall:.4f} | F1-Score: {f1:.4f}')
print(f'Taxa de Falsos Positivos (FPR): {fp_rate:.2%}')
print('Matriz de Confusão (TN, FP / FN, TP):\n', cm)
print('\n=== DISTRIBUIÇÃO DE AÇÕES DO PLAYBOOK SHUFFLE SOAR ===')
print(df['soar_action'].value_counts().to_string())


=== RESULTADOS DA EXECUÇÃO DO PROTÓTIPO (ISOLATION FOREST + SOAR) ===
Total de eventos processados: 2080 (Normais: 2000 | Ataques: 80)
Precision: 0.8696 | Recall: 1.0000 | F1-Score: 0.9302
Taxa de Falsos Positivos (FPR): 0.60%
Matriz de Confusão (TN, FP / FN, TP):
[[1988   12]
 [   0   80]]

=== DISTRIBUIÇÃO DE AÇÕES DO PLAYBOOK SHUFFLE SOAR ===
soar_action
BAIXO -> Log ECS & Atualizacao Baseline                                         1988
ALTO/CRITICO -> Bloqueio WAF (TTL 24h) + Revogar Token OAuth2 + Caso TheHive      81
MEDIO -> Rate Limiting Adaptativo + Step-Up Auth (MFA/CAPTCHA)                    11

=== PAYLOAD ECS ENVIADO AO WEBHOOK SOAR ===
{
  "@timestamp": "2026-11-24T03:14:22.418Z",
  "event": {
    "kind": "alert",
    "category": [
      "web",
      "authentication"
    ],
    "action": "api_anomaly_detected"
  },
  "ml": {
    "model": "isolation_forest_v2.4",
    "anomaly_score": 0.8847,
    "features": {
      "off_hours_flag": 1,
      "req_rate_1m": 22.92,
      "